In [12]:
import pandas as pd
import ast
import gender_guesser.detector as gender

In [13]:
# 1. DEFINICIÓN DE VARIABLES
d = gender.Detector(case_sensitive=False)
archivo_entrada = 'dataset_analisis_corpus.csv'
archivo_csv = 'dataset_con_genero_final.csv'
archivo_pickle = 'dataset_con_genero_final.pkl' # Nueva extensión correcta

In [14]:
# 2. CARGA GLOBAL
df = pd.read_csv(archivo_entrada)

def identificar_genero(nombre):
    nombre_clean = str(nombre).upper().strip()
    palabras = nombre_clean.split()
    
    # Heurística de palabras clave (Prioridad 1)
    feminine = ['MOTHER', 'WOMAN', 'GIRL', 'MRS', 'MISS', 'MS', 'SISTER', 'LADY', 'NURSE', 'WAITRESS']
    masculine = ['FATHER', 'MAN', 'BOY', 'MR', 'SIR', 'LORD', 'SON', 'WAITER', 'BROTHER', 'GUY']
    
    if any(w in palabras for w in feminine): return 'female'
    if any(w in palabras for w in masculine): return 'male'
    
    # Manejo de Títulos como "DR." (Prioridad 2)
    # Si el nombre empieza por DR., PROF., etc., intentamos analizar la segunda palabra
    titulos_neutros = ['DR', 'DR.', 'PROF', 'PROF.', 'COL', 'CAPT', 'AGENT']
    if palabras[0] in titulos_neutros and len(palabras) > 1:
        pila = palabras[1].capitalize()
    else:
        pila = palabras[0].capitalize()
        
    resultado = d.get_gender(pila)
    
    # Si el detector falla (como con "HOLTZ"), pero el usuario sabe que ciertos 
    # roles suelen ser masculinos en este corpus, puedes añadir excepciones:
    if 'female' in resultado: return 'female'
    if 'male' in resultado: return 'male'
    
    return 'unknown'

def procesar_columna_script():
    lista_resultados = []
    for index, row in df.iterrows():
        diccionario_generos = {}
        script_dict_str = row['Script_Dict']
        if pd.notna(script_dict_str):
            try:
                # Limpieza de saltos de línea para evitar SyntaxError / EOL
                limpio = script_dict_str.replace('\n', ' ').replace('\r', ' ')
                script_data = ast.literal_eval(limpio)
                for personaje in script_data.keys():
                    diccionario_generos[personaje] = identificar_genero(personaje)
            except:
                pass 
        lista_resultados.append(diccionario_generos)
    return lista_resultados

In [15]:
# 3. EJECUCIÓN
print("Procesando géneros... esto puede tardar unos segundos.")
df['Character_Genders'] = procesar_columna_script()

Procesando géneros... esto puede tardar unos segundos.


In [16]:
# 4. GUARDADO CORRECTO
# CSV acepta index=False
df.to_csv(archivo_csv, index=False)
df.to_pickle(archivo_pickle)

In [17]:
print(f"✅ ¡Hecho!")
print(f"Archivo CSV: {archivo_csv}")
print(f"Archivo Pickle: {archivo_pickle}")

✅ ¡Hecho!
Archivo CSV: dataset_con_genero_final.csv
Archivo Pickle: dataset_con_genero_final.pkl
